# MScFE 660 Risk Management: Group Work Project #2

**Methodology Description and Model Development: Hidden Markov Regimes and a Bayesian Network of Crude Oil Prices**

**Team Members (Student Group 18596)**
- Vyom Vijaybhai Modi
- Sushrut Dilip Shendre
- Olukayode Olusegun Odukoya

**Objective.** This notebook is the computational companion to the GWP2 report and continues the GWP1 work on the same data. Following Alvi (2018), the price of oil and its drivers are reduced to three hidden regimes each with a Hidden Markov Model (HMM), and the regimes are then connected in a Bayesian network whose structure is learned by hill climbing with `pgmpy`. Step 1 implements the four HMM algorithms from scratch and tests them on a seeded toy example, Step 2 identifies bull, bear and stagnant regimes in the WTI price, Step 4 runs Alvi's pipeline of Figure 3.1 on the sterilized GWP1 panel (regime process, network training, parameter testing and validation), Steps 6 and 7 document how the parameters are learned, how the most likely regime path is decoded and what each hidden state means, and Step 9 gives the minimal working example of hill climbing.

**Division of labour.**

| Role | Member | Step 1 (algorithm) | Step 2 (regime) | Step 4 (pipeline) | Step 6 (regime detection) |
|---|---|---|---|---|---|
| Student A | Sushrut Dilip Shendre | Forward and backward | Bull | Macro research, indicators, retrieval, cleaning | Transforming the time series |
| Student B | Olukayode Olusegun Odukoya | Viterbi | Bear | Regime process and the `hmms` code | How the parameters are learned |
| Student C | Vyom Vijaybhai Modi | Baum-Welch | Stagnant | Training with `pgmpy`, testing and validation | The most likely hidden-state sequence |

**Data.** The notebook starts from the sterilized monthly panel produced in GWP1 (`monthly_sterilized.csv`, 41 series from January 2005 to July 2026, built from FRED, EIA, Yahoo Finance and the Caldara and Iacoviello GPR index without any API key) so that every number in GWP2 rests on exactly the data the GWP1 report describes.

## Setup and Reproducibility

A single random seed (18596, the group number) governs every stochastic routine: the toy HMM generator, the random starting parameters of every Baum-Welch run, the bootstrap resamples of the training months, and the layout of the network figures. All other computations are deterministic. Every table is saved as a CSV file and every figure as a PNG file inside `GWP2_Outputs/`, zipped at the end so the report can reuse them.

In [ ]:
# ---------------------------------------------------------------- environment
import sys, subprocess, importlib

def _ensure(package, module=None):
    """Install a package quietly only if it is not already importable."""
    try:
        importlib.import_module(module or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

for _pkg, _mod in [("pgmpy", "pgmpy"), ("hmmlearn", "hmmlearn"), ("networkx", "networkx")]:
    _ensure(_pkg, _mod)

import os, io, math, random, warnings, zipfile, itertools, time
from datetime import date

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns
import networkx as nx
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 110)

# ---------------------------------------------------------------- reproducibility
SEED = 18596
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

print("=" * 70)
print("REPRODUCIBILITY")
print(f"Random seed fixed at {SEED} for numpy, random and the Generator RNG.")
print("Data: the GWP1 sterilized monthly panel (January 2005 to July 2026).")
print("Re-running the notebook reproduces every output.")
print("=" * 70)

# ---------------------------------------------------------------- house style
WQU_PURPLE = "#3B3273"
LAVENDER = "#8C84C9"
LILAC = "#C9C3E6"
GREY = "#8A8A8A"
AMBER = "#D98C21"
TEAL = "#2A9D8F"
INK = "#2B2B2B"
PALETTE = [WQU_PURPLE, AMBER, LAVENDER, GREY, TEAL, LILAC]
REGIME_COLORS = {0: AMBER, 1: GREY, 2: WQU_PURPLE}      # bear, stagnant, bull
REGIME_NAMES = {0: "Bear", 1: "Stagnant", 2: "Bull"}

sns.set_theme(style="whitegrid", context="notebook")
mpl.rcParams.update({
    "axes.prop_cycle": mpl.cycler(color=PALETTE),
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "axes.titleweight": "bold", "axes.titlecolor": WQU_PURPLE, "axes.titlesize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK, "ytick.color": INK,
    "grid.color": "#ECEAF4", "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 9,
    "font.size": 10, "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.titleweight": "bold", "figure.titlesize": 13,
})
PURPLE_CMAP = sns.light_palette(WQU_PURPLE, as_cmap=True)

# ---------------------------------------------------------------- output helpers
OUT = "GWP2_Outputs"
FIG_NO = 0
TAB_NO = 0

def outdir(step):
    path = os.path.join(OUT, f"Step{step}")
    os.makedirs(path, exist_ok=True)
    return path

def show_fig(fig, step, slug, caption):
    """Save the figure, display it, and print its running number and caption."""
    global FIG_NO
    FIG_NO += 1
    path = os.path.join(outdir(step), f"Fig{FIG_NO:02d}_{slug}.png")
    fig.savefig(path)
    plt.show()
    plt.close(fig)
    print(f"Fig. {FIG_NO}  {caption}   [saved to {path}]")

def show_table(df, step, slug, caption, decimals=4):
    """Save the table as CSV, display it, and print its running number and caption."""
    global TAB_NO
    TAB_NO += 1
    path = os.path.join(outdir(step), f"Table{TAB_NO:02d}_{slug}.csv")
    df.to_csv(path)
    print(f"Table {TAB_NO}. {caption}   [saved to {path}]")
    display(df.round(decimals) if decimals is not None else df)

# Major oil market events used for shading time series plots
EVENTS = [
    ("2008-09-01", "2009-03-31", "Global financial crisis"),
    ("2014-11-01", "2016-02-29", "OPEC market share war"),
    ("2020-03-01", "2020-05-31", "COVID demand collapse"),
    ("2022-02-24", "2022-06-30", "Russia invades Ukraine"),
]

def shade_events(ax, label=False):
    for i, (s, e, lab) in enumerate(EVENTS):
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=LILAC, alpha=0.4, lw=0,
                   label="Major oil market events" if (label and i == 0) else None)

import pgmpy, hmmlearn
print("Environment ready. pandas", pd.__version__, "| numpy", np.__version__,
      "| pgmpy", pgmpy.__version__, "| hmmlearn", hmmlearn.__version__)

## Data: The GWP1 Sterilized Panel and Alvi's Split

GWP1 identified, retrieved and cleaned 49 variables (Step 4a of this project is the written account of that process). Two files from the GWP1 notebook are the inputs here:

- `monthly_sterilized.csv`: the sterilized monthly panel (41 series, January 2005 to July 2026), with weekend stamps dropped, holiday blanks forward filled, the April 2020 negative price kept, and the common sample applied;
- `steo_world_block.csv`: the EIA Short-Term Energy Outlook world balance (OPEC and non-OPEC supply, OECD and non-OECD demand, inventories, spare capacity), which the public workbook only provides from January 2022 and which is therefore too short for a regime model (it is kept for the report's description of Alvi's variables).

Two sets of variables are carried through the notebook:

- **Alvi-like set**: the physical and macro series that correspond to Alvi's EIA and FRED choices (real GDP, CPI, industrial production, oil and gas capacity utilisation, drilling, U.S. crude production, stocks and imports, refinery utilisation, the crude PPI, the WTI spot price);
- **GWP1 set**: the sixteen variables of the GWP1 Inferred Causality graph (WTI, SPX, DXY, gold, OVX, the ten-year yield, GPR, GEPU, industrial production, CPI, the funds rate, crude production, stocks, refinery utilisation, drilling, gasoline demand).

Both sets receive the target node `WTI_next`, the regime of the WTI price one month ahead, which plays the role of Alvi's `forecast` column. The sample is split by time in Alvi's 80:10:10 proportions into training, validation and test blocks. Only the training block is used to learn HMM parameters and network structure; the validation block tunes the choices; the test block is touched once at the end.

In [ ]:
DATA_FILES = ["monthly_sterilized.csv", "steo_world_block.csv"]
# Web copies of the two GWP1 files (raw GitHub URLs). Leave empty to use local files or a Colab upload.
DATA_URL_BASE = "https://raw.githubusercontent.com/vyom-modi/wqu-rm-gwp2/main"

def _locate(name):
    for folder in [".", "data", "GWP1_Colab_Outputs/Step6", "/content", "/content/data"]:
        p = os.path.join(folder, name)
        if os.path.exists(p):
            return p
    return None

paths = {f: _locate(f) for f in DATA_FILES}
if DATA_URL_BASE and any(v is None for v in paths.values()):
    import requests
    for f in [f for f, p in paths.items() if p is None]:
        r = requests.get(DATA_URL_BASE.rstrip("/") + "/" + f, timeout=60)
        r.raise_for_status()
        with open(f, "wb") as fh:
            fh.write(r.content)
        print(f"Downloaded {f} ({len(r.content) / 1e3:.0f} kB) from {DATA_URL_BASE}")
    paths = {f: _locate(f) for f in DATA_FILES}
if any(v is None for v in paths.values()):
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None
    if colab_files is not None:
        for _attempt in range(3):                      # one prompt per missing file; select several at once if you like
            still = [f for f, p in paths.items() if p is None]
            if not still:
                break
            print(f"Upload {still} from the GWP1 outputs (Step6). You can select both files in one dialog.")
            colab_files.upload()
            paths = {f: _locate(f) for f in DATA_FILES}
missing = [f for f, p in paths.items() if p is None]
assert not missing, f"Missing data files: {missing}. Put them next to the notebook, in ./data/, or set DATA_URL_BASE."

monthly = pd.read_csv(paths["monthly_sterilized.csv"], index_col=0, parse_dates=True).sort_index()
steo = pd.read_csv(paths["steo_world_block.csv"], index_col=0, parse_dates=True).sort_index()
print(f"Monthly panel: {monthly.shape[1]} series x {monthly.shape[0]} months, {monthly.index.min():%b %Y} to {monthly.index.max():%b %Y}")
print(f"STEO world block: {steo.shape[1]} series x {steo.shape[0]} months, {steo.index.min():%b %Y} to {steo.index.max():%b %Y}")

ALVI_LIKE = ["RealGDP", "CPI", "INDPRO", "CapUtil_oilgas", "Drilling_idx", "US_crude_prod",
             "US_crude_stocks", "US_crude_imports", "Refinery_util", "PPI_crude", "Gasoline_demand", "WTI_spot"]
GWP1_NODES = ["WTI_spot", "SPX", "DXY", "Gold_fut", "OVX", "UST10Y", "GPR", "GEPU", "INDPRO", "CPI", "FedFunds",
              "US_crude_prod", "US_crude_stocks", "Refinery_util", "Drilling_idx", "Gasoline_demand"]
ALL_VARS = list(dict.fromkeys(GWP1_NODES + ALVI_LIKE))

# Alvi's rule: the previous month's level is needed, so the first observation is lost to the difference.
levels = monthly[ALL_VARS].dropna()
changes = levels.diff().iloc[1:]
levels = levels.iloc[1:]

# Stepped series (quarterly GDP repeated within the quarter) would emit "no change" two months in three.
# For them the quarterly change is carried through the quarter so that every month has a direction.
stepped = [c for c in changes.columns if (changes[c] == 0).mean() > 0.3]
for c in stepped:
    changes[c] = changes[c].replace(0, np.nan).ffill().fillna(0)
print("Stepped series with the change carried through the period:", stepped)

n = len(changes)
n_train, n_val = int(0.8 * n), int(0.9 * n) - int(0.8 * n)
n_test = n - n_train - n_val
idx_train = changes.index[:n_train]
idx_val = changes.index[n_train:n_train + n_val]
idx_test = changes.index[n_train + n_val:]
split_tbl = pd.DataFrame({
    "block": ["training", "validation", "test"],
    "months": [n_train, n_val, n_test],
    "from": [idx_train[0].strftime("%b %Y"), idx_val[0].strftime("%b %Y"), idx_test[0].strftime("%b %Y")],
    "to": [idx_train[-1].strftime("%b %Y"), idx_val[-1].strftime("%b %Y"), idx_test[-1].strftime("%b %Y")],
}).set_index("block")
show_table(split_tbl, 4, "sample_split", f"Alvi's 80:10:10 split by time of the {n}-month common sample of the {len(ALL_VARS)} variables", decimals=None)

desc = pd.DataFrame({
    "first month": levels.index[0].strftime("%b %Y"),
    "months": n,
    "mean monthly change": changes.mean(),
    "sd of monthly change": changes.std(),
    "share of months up": (changes > 0).mean(),
})
show_table(desc, 4, "variables_in_scope", "The variables carried through GWP2 and the direction of their monthly changes on the common sample", decimals=3)

## Step 1: The Four Hidden Markov Model Algorithms, Written from Scratch

Alvi's Definition 3 gives an HMM as $\lambda = (\Pi, A, B)$ over $N$ hidden states and $M$ symbols: $\pi_i$ is the probability of starting in state $i$, $a_{ij}$ the probability of moving from state $i$ to state $j$, and $b_i(k)$ the probability that state $i$ emits symbol $k$. The three classical problems and their solutions are implemented below in plain `numpy`, with the pseudocode written in the report reproduced as the docstring of each method. The `hmms` library that Alvi used wraps the same four routines; the lesson's `hmmlearn` is used as an independent check.

- **Forward algorithm (Student A).** $\alpha_t(i) = P(o_1, \dots, o_t, q_t = i \mid \lambda)$, computed by induction $\alpha_{t+1}(j) = b_j(o_{t+1}) \sum_i \alpha_t(i) a_{ij}$, scaled at every step so that long sequences do not underflow; the likelihood is $P(O \mid \lambda) = \sum_i \alpha_T(i)$, recovered from the scaling constants as $\log P = \sum_t \log c_t$.
- **Backward algorithm (Student A).** $\beta_t(i) = P(o_{t+1}, \dots, o_T \mid q_t = i, \lambda)$, computed backwards from $\beta_T(i) = 1$ with $\beta_t(i) = \sum_j a_{ij} b_j(o_{t+1}) \beta_{t+1}(j)$ and the same scaling. Together, $\gamma_t(i) \propto \alpha_t(i)\beta_t(i)$ is the posterior probability of being in state $i$ at time $t$.
- **Viterbi algorithm (Student B).** The most likely state path $q^* = \arg\max_q P(q, O \mid \lambda)$ by dynamic programming in log space: $\delta_{t+1}(j) = \max_i [\delta_t(i) + \log a_{ij}] + \log b_j(o_{t+1})$ with back-pointers $\psi$, then a backward trace from the best final state.
- **Baum-Welch algorithm (Student C).** Expectation Maximisation: the E step computes $\gamma_t(i)$ and $\xi_t(i, j) \propto \alpha_t(i) a_{ij} b_j(o_{t+1}) \beta_{t+1}(j)$ from the forward and backward passes; the M step re-estimates $\pi_i = \gamma_1(i)$, $a_{ij} = \sum_t \xi_t(i,j) / \sum_t \gamma_t(i)$ and $b_j(k) = \sum_{t: o_t = k} \gamma_t(j) / \sum_t \gamma_t(j)$. Counts are pooled over several sequences, which is why Alvi could split his series into pieces of at most 32 observations. Each iteration raises the likelihood; the search is local, so several random starts are run and the best kept. The implementation also accepts a sticky prior, $\kappa$ pseudo-counts added to the diagonal of $A$ in the M step (Fox et al., 2011), which is switched off ($\kappa = 0$) for Alvi's exact method and used in Step 2 to keep regimes from flipping every month.

In [ ]:
class DiscreteHMM:
    """A discrete-emission HMM lambda = (pi, A, B) with the four classical algorithms."""

    def __init__(self, n_states, n_symbols, rng=None, pi=None, A=None, B=None):
        self.N, self.M = n_states, n_symbols
        rng = rng if rng is not None else np.random.default_rng()
        self.pi = pi if pi is not None else rng.dirichlet(np.ones(n_states))
        self.A = A if A is not None else rng.dirichlet(np.ones(n_states), size=n_states)
        self.B = B if B is not None else rng.dirichlet(np.ones(n_symbols), size=n_states)
        self.history = []

    # ------------------------------------------------------------ evaluation
    def forward(self, obs):
        """Scaled forward pass.
        alpha_1(i) = pi_i * b_i(o_1);  alpha_t+1(j) = b_j(o_t+1) * sum_i alpha_t(i) a_ij
        Each alpha_t is divided by c_t = sum_i alpha_t(i); log P(O | lambda) = sum_t log c_t."""
        T = len(obs)
        alpha = np.zeros((T, self.N))
        c = np.zeros(T)
        alpha[0] = self.pi * self.B[:, obs[0]]
        c[0] = alpha[0].sum()
        alpha[0] /= c[0]
        for t in range(1, T):
            alpha[t] = (alpha[t - 1] @ self.A) * self.B[:, obs[t]]
            c[t] = alpha[t].sum()
            alpha[t] /= c[t]
        return alpha, c, float(np.log(c).sum())

    def backward(self, obs, c):
        """Scaled backward pass.
        beta_T(i) = 1;  beta_t(i) = sum_j a_ij b_j(o_t+1) beta_t+1(j), scaled by the forward c_t+1."""
        T = len(obs)
        beta = np.zeros((T, self.N))
        beta[T - 1] = 1.0
        for t in range(T - 2, -1, -1):
            beta[t] = (self.A @ (self.B[:, obs[t + 1]] * beta[t + 1])) / c[t + 1]
        return beta

    def posterior(self, obs):
        """gamma_t(i) = P(q_t = i | O, lambda), from the scaled forward and backward passes."""
        alpha, c, ll = self.forward(obs)
        beta = self.backward(obs, c)
        gamma = alpha * beta
        return gamma / gamma.sum(axis=1, keepdims=True), ll

    # ------------------------------------------------------------ decoding
    def viterbi(self, obs):
        """Most likely state path in log space.
        delta_1(i) = log pi_i + log b_i(o_1)
        delta_t+1(j) = max_i [delta_t(i) + log a_ij] + log b_j(o_t+1), psi_t+1(j) = argmax_i
        q*_T = argmax_i delta_T(i); q*_t = psi_t+1(q*_t+1)."""
        T = len(obs)
        with np.errstate(divide="ignore"):
            lpi, lA, lB = np.log(self.pi), np.log(self.A), np.log(self.B)
        delta = np.zeros((T, self.N))
        psi = np.zeros((T, self.N), dtype=int)
        delta[0] = lpi + lB[:, obs[0]]
        for t in range(1, T):
            cand = delta[t - 1][:, None] + lA
            psi[t] = cand.argmax(axis=0)
            delta[t] = cand.max(axis=0) + lB[:, obs[t]]
        path = np.zeros(T, dtype=int)
        path[-1] = delta[-1].argmax()
        for t in range(T - 2, -1, -1):
            path[t] = psi[t + 1, path[t + 1]]
        return path, float(delta[-1].max())

    # ------------------------------------------------------------ training
    def baum_welch(self, seqs, n_iter=100, tol=1e-6, kappa=0.0, verbose=False):
        """Expectation Maximisation over one or more observation sequences.
        E step: gamma_t(i) = alpha_t(i) beta_t(i) / sum_i, xi_t(i,j) = alpha_t(i) a_ij b_j(o_t+1) beta_t+1(j) / c_t+1
        M step: pi_i = mean over sequences of gamma_1(i); a_ij = (sum_t xi_t(i,j) + kappa [i = j]) / (sum_t gamma_t(i) + kappa);
                b_j(k) = sum_{t: o_t = k} gamma_t(j) / sum_t gamma_t(j).
        kappa = 0 is plain maximum likelihood; kappa > 0 adds kappa pseudo-counts of "stay in the same state"
        (a sticky Dirichlet prior on each row of A, the MAP variant of Fox et al. 2011).
        Stops when the pooled log-likelihood improves by less than tol."""
        seqs = [np.asarray(s, dtype=int) for s in seqs if len(s) > 1]
        self.history = []
        for it in range(n_iter):
            pi_acc = np.zeros(self.N)
            A_num = np.zeros((self.N, self.N))
            A_den = np.zeros(self.N)
            B_num = np.zeros((self.N, self.M))
            B_den = np.zeros(self.N)
            total_ll = 0.0
            for obs in seqs:
                alpha, c, ll = self.forward(obs)
                beta = self.backward(obs, c)
                total_ll += ll
                gamma = alpha * beta
                gamma /= gamma.sum(axis=1, keepdims=True)
                # xi summed over t, computed in one vectorised expression
                xi_sum = np.zeros((self.N, self.N))
                for t in range(len(obs) - 1):
                    xi = (alpha[t][:, None] * self.A) * (self.B[:, obs[t + 1]] * beta[t + 1])[None, :] / c[t + 1]
                    xi_sum += xi
                pi_acc += gamma[0]
                A_num += xi_sum
                A_den += gamma[:-1].sum(axis=0)
                for k in range(self.M):
                    B_num[:, k] += gamma[obs == k].sum(axis=0)
                B_den += gamma.sum(axis=0)
            self.history.append(total_ll)
            self.pi = pi_acc / len(seqs)
            self.A = (A_num + kappa * np.eye(self.N)) / np.maximum(A_den + kappa, 1e-300)[:, None]
            self.B = B_num / np.maximum(B_den, 1e-300)[:, None]
            self.A /= self.A.sum(axis=1, keepdims=True)
            self.B /= self.B.sum(axis=1, keepdims=True)
            if verbose and (it % 10 == 0):
                print(f"iteration {it:3d}: log-likelihood {total_ll:.4f}")
            if it > 0 and abs(self.history[-1] - self.history[-2]) < tol:
                break
        return self

    def loglik(self, seqs):
        return float(sum(self.forward(np.asarray(s, dtype=int))[2] for s in seqs if len(s) > 1))

    def sample(self, T, rng):
        """Generate a state path and an emission sequence of length T."""
        states = np.zeros(T, dtype=int)
        obs = np.zeros(T, dtype=int)
        states[0] = rng.choice(self.N, p=self.pi)
        obs[0] = rng.choice(self.M, p=self.B[states[0]])
        for t in range(1, T):
            states[t] = rng.choice(self.N, p=self.A[states[t - 1]])
            obs[t] = rng.choice(self.M, p=self.B[states[t]])
        return states, obs


def fit_best(seqs, n_states, n_symbols, n_restarts=20, n_iter=100, kappa=0.0, rng=None):
    """Baum-Welch from several random starts; returns the model with the highest log-likelihood and all final log-likelihoods."""
    rng = rng if rng is not None else RNG
    best, lls = None, []
    for r in range(n_restarts):
        m = DiscreteHMM(n_states, n_symbols, rng=rng).baum_welch(seqs, n_iter=n_iter, kappa=kappa)
        ll = m.loglik(seqs)
        lls.append(ll)
        if best is None or ll > best.loglik(seqs):
            best = m
    return best, np.array(lls)


def match_states(est, true):
    """Permutation of the estimated states that best matches the true emission rows (label switching)."""
    best_perm, best_err = None, np.inf
    for perm in itertools.permutations(range(true.N)):
        err = np.abs(est.B[list(perm)] - true.B).sum()
        if err < best_err:
            best_perm, best_err = perm, err
    return list(best_perm)

print("DiscreteHMM defined: forward, backward, posterior, viterbi, baum_welch, sample.")

### Toy example

A three-state, three-symbol HMM with known parameters stands in for a market that is in a bear, a stagnant or a bull regime and each month emits a "large fall" (0), a "small move" (1) or a "large rise" (2); regimes persist with probability 0.9 per month. The generator is seeded, so the hidden path and the observed sequence are the same on every run. The same toy serves the three students: the forward and backward passes give the posterior probability of each regime at each month (A), the Viterbi path is compared with the true hidden path (B), and Baum-Welch is run from random starts to see how closely it recovers $\Pi$, $A$ and $B$ (C). The sequence is fitted whole and, as a second run, cut into pieces of at most 32 symbols as the `hmms` package that Alvi used requires, to show what that constraint costs: every piece is treated as a fresh start from $\Pi$, so the transition counts lose one observation per piece and $\Pi$ is estimated from the piece starts rather than from the single true start.

In [ ]:
PI_TRUE = np.array([0.25, 0.50, 0.25])
A_TRUE = np.array([[0.90, 0.07, 0.03],
                   [0.05, 0.90, 0.05],
                   [0.03, 0.07, 0.90]])
B_TRUE = np.array([[0.70, 0.25, 0.05],     # bear: mostly large falls
                   [0.20, 0.60, 0.20],     # stagnant: mostly small moves
                   [0.05, 0.25, 0.70]])    # bull: mostly large rises
M_TOY = B_TRUE.shape[1]
true_model = DiscreteHMM(3, M_TOY, pi=PI_TRUE, A=A_TRUE, B=B_TRUE)
T_TOY = 1000
toy_states, toy_obs = true_model.sample(T_TOY, RNG)
print(f"Toy sequence of {T_TOY} symbols; true state shares:", np.bincount(toy_states, minlength=3) / T_TOY)

# ---- Student A: forward and backward
alpha, c, ll_true = true_model.forward(toy_obs)
gamma, _ = true_model.posterior(toy_obs)
from hmmlearn.hmm import CategoricalHMM
chk = CategoricalHMM(n_components=3, n_features=M_TOY, init_params="")
chk.startprob_, chk.transmat_, chk.emissionprob_ = PI_TRUE, A_TRUE, B_TRUE
ll_hmmlearn = chk.score(toy_obs.reshape(-1, 1))
print(f"log P(O | lambda_true): our forward {ll_true:.6f} | hmmlearn {ll_hmmlearn:.6f} | difference {abs(ll_true - ll_hmmlearn):.2e}")
post_acc = (gamma.argmax(axis=1) == toy_states).mean()

fig, ax = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
ax[0].step(range(T_TOY), toy_states, where="mid", color=INK, lw=1)
ax[0].set_yticks([0, 1, 2]); ax[0].set_yticklabels(["Bear", "Stagnant", "Bull"]); ax[0].set_title("True hidden path")
ax[1].stackplot(range(T_TOY), gamma.T, colors=[AMBER, GREY, WQU_PURPLE], labels=["Bear", "Stagnant", "Bull"], alpha=0.85)
ax[1].set_ylim(0, 1); ax[1].set_xlim(0, 200); ax[1].set_xlabel(f"time (first 200 of {T_TOY} symbols shown)")
ax[1].set_title("Posterior regime probabilities from the forward and backward passes"); ax[1].legend(loc="upper right", ncol=3)
show_fig(fig, 1, "toy_forward_backward_posterior", "Toy example: the true hidden path and the posterior probabilities gamma_t(i) from the forward and backward passes")
print(f"Share of months where the most probable posterior state equals the true state: {post_acc:.3f}")

# ---- Student B: Viterbi
vpath, vscore = true_model.viterbi(toy_obs)
vit_acc = (vpath == toy_states).mean()
hl_path = chk.predict(toy_obs.reshape(-1, 1))
print(f"Viterbi path agrees with the true path on {vit_acc:.3f} of the months and with hmmlearn's decode on {(vpath == hl_path).mean():.3f} (differences are tied paths)")

# ---- Student C: Baum-Welch from random starts on the whole sequence
t0 = time.time()
toy_fit, toy_lls = fit_best([toy_obs], 3, M_TOY, n_restarts=20, n_iter=300)
perm = match_states(toy_fit, true_model)
pi_hat, A_hat, B_hat = toy_fit.pi[perm], toy_fit.A[np.ix_(perm, perm)], toy_fit.B[perm]
print(f"Baum-Welch: 20 random starts x up to 300 iterations in {time.time() - t0:.1f} s; best log-likelihood {toy_lls.max():.3f}, worst {toy_lls.min():.3f}, at the true parameters {ll_true:.3f}")

# The same fit on pieces of at most 32 symbols, the constraint of Alvi's hmms package
toy_pieces = np.array_split(toy_obs, int(np.ceil(T_TOY / 32)))
toy_fit_pieces, _ = fit_best(toy_pieces, 3, M_TOY, n_restarts=20, n_iter=300)
perm_p = match_states(toy_fit_pieces, true_model)
err_pieces = max(np.abs(toy_fit_pieces.A[np.ix_(perm_p, perm_p)] - A_TRUE).max(), np.abs(toy_fit_pieces.B[perm_p] - B_TRUE).max())
err_AB = max(np.abs(A_hat - A_TRUE).max(), np.abs(B_hat - B_TRUE).max())

rows = []
for name, true_v, est_v in [("pi", PI_TRUE, pi_hat), ("A", A_TRUE, A_hat), ("B", B_TRUE, B_hat)]:
    for idx in np.ndindex(true_v.shape):
        lab = name + "_" + "".join(str(i + 1) for i in idx)
        rows.append({"parameter": lab, "true": true_v[idx], "estimated": est_v[idx], "abs error": abs(true_v[idx] - est_v[idx])})
toy_tbl = pd.DataFrame(rows).set_index("parameter")
show_table(toy_tbl, 1, "toy_baum_welch_recovery", "Toy example: true parameters against the Baum-Welch estimates (states matched by their emission rows)", decimals=3)
print(f"Largest absolute error in A and B: {err_AB:.3f}; mean absolute error over all parameters: {toy_tbl['abs error'].mean():.3f}")
print(f"The initial vector pi is a one-sample estimate (one sequence has one start), so its error of {np.abs(pi_hat - PI_TRUE).max():.2f} is expected and uninformative.")
print(f"Largest absolute error in A and B when the sequence is cut into {len(toy_pieces)} pieces of at most 32 symbols: {err_pieces:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(toy_fit.history, color=WQU_PURPLE)
ax.axhline(ll_true, color=AMBER, ls="--", label="log-likelihood at the true parameters")
ax.set_xlabel("Baum-Welch iteration"); ax.set_ylabel("log-likelihood"); ax.legend()
ax.set_title("Baum-Welch convergence on the toy sequence (best of 20 starts)")
show_fig(fig, 1, "toy_baum_welch_convergence", "Toy example: pooled log-likelihood per Baum-Welch iteration for the best random start")

# The fitted model decodes the toy sequence nearly as well as the true one
fit_path, _ = toy_fit.viterbi(toy_obs)
fit_path = np.array(perm).argsort()[fit_path]       # relabel to the true ordering
toy_summary = pd.DataFrame({
    "quantity": ["posterior argmax accuracy (true parameters)", "Viterbi accuracy (true parameters)",
                 "Viterbi accuracy (Baum-Welch parameters)", "random starts within 1 log-likelihood unit of the best",
                 "largest error in A and B, whole sequence", "largest error in A and B, pieces of 32"],
    "value": [post_acc, vit_acc, (fit_path == toy_states).mean(), (toy_lls > toy_lls.max() - 1).mean(),
              err_AB, err_pieces],
}).set_index("quantity")
show_table(toy_summary, 1, "toy_summary", "Toy example: decoding accuracy and the stability of Baum-Welch across random starts", decimals=3)

## Step 2: Bull, Bear and Stagnant Regimes in the WTI Price

Alvi's regime detection (Section 4.2) turns the monthly WTI series into a sequence of symbols, 1 when the price rose on the month and 0 when it fell, fits a three-state HMM to the training block with 100 Baum-Welch iterations from one random start, and names the states by the average price change under each: the lowest mean is the bear regime, the highest the bull regime, the middle one stagnant. That procedure is run first, with two repairs that do not change its logic: the fit is repeated from 20 random starts and the best log-likelihood is kept, because Baum-Welch is a local search; and the state naming sorts the state means, so the mapping cannot be scrambled, which is what the dictionary in Alvi's notebook does when the sorted and unsorted orders differ. The validation and test months are decoded separately with the training parameters, as Alvi does, so no future information enters the training block.

In [ ]:
N_SYMBOLS = {"sign": 2, "quintile": 5}
SYMBOL_NAMES = {"sign": ["down", "up"], "quintile": ["large fall", "fall", "small move", "rise", "large rise"]}
KAPPA_GWP2 = 5.0

def to_symbols(chg, kind, thresholds=None):
    """Monthly changes to symbols. 'sign': 1 up, 0 down (Alvi). 'quintile': 0 to 4 by the training quintiles of the change."""
    if kind == "sign":
        return (chg > 0).astype(int).values, None
    cuts = thresholds if thresholds is not None else np.quantile(chg, [0.2, 0.4, 0.6, 0.8])
    return np.searchsorted(cuts, np.asarray(chg), side="right"), cuts

def label_by_mean(states, chg):
    """Map raw state ids to 0 bear, 1 stagnant, 2 bull by the mean change in each state."""
    means = pd.Series(chg).groupby(states).mean()
    order = means.sort_values().index.tolist()          # lowest mean first
    while len(order) < 3:                                # a state never visited keeps a slot
        order.append([s for s in range(3) if s not in order][0])
    return {raw: rank for rank, raw in enumerate(order)}

def detect_regimes(chg, kind="sign", kappa=0.0, n_restarts=20, n_iter=100, piece=None, rng=None):
    """Fit on the training block, decode the three blocks separately; return states (full index), model, info.
    piece=32 reproduces Alvi's split of the training sequence into pieces (the hmms constraint); None fits it whole."""
    rng = rng if rng is not None else RNG
    tr = chg.loc[idx_train]
    sym_tr, thr = to_symbols(tr, kind)
    M = N_SYMBOLS[kind]
    pieces = np.array_split(sym_tr, int(np.ceil(len(sym_tr) / piece))) if piece else [sym_tr]
    model, lls = fit_best(pieces, 3, M, n_restarts=n_restarts, n_iter=n_iter, kappa=kappa, rng=rng)
    raw_tr, _ = model.viterbi(sym_tr)
    mapping = label_by_mean(raw_tr, tr.values)
    states = pd.Series(index=chg.index, dtype=float)
    for block in [idx_train, idx_val, idx_test]:
        sym, _ = to_symbols(chg.loc[block], kind, thr)
        raw, _ = model.viterbi(sym)
        states.loc[block] = [mapping[r] for r in raw]
    states = states.astype(int)
    perm = sorted(mapping, key=mapping.get)             # raw ids in bear, stagnant, bull order
    info = {"loglik_best": lls.max(), "loglik_spread": lls.max() - lls.min(),
            "starts_within_1": (lls > lls.max() - 1).mean(), "A": model.A[np.ix_(perm, perm)],
            "B": model.B[perm], "pi": model.pi[perm], "history": model.history, "thresholds": thr}
    return states, model, info

def regime_plot(ax, series, states, title, ylabel=""):
    """Alvi's multicoloured time series: the line takes the colour of the regime of each month."""
    x = mdates.date2num(series.index.to_pydatetime())
    pts = np.array([x, series.values]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap=ListedColormap([AMBER, GREY, WQU_PURPLE]), norm=BoundaryNorm(range(4), 3), lw=1.8)
    lc.set_array(states.values[:-1])
    ax.add_collection(lc)
    ax.set_xlim(series.index.min(), series.index.max())
    ax.set_ylim(series.min() * 0.98 if series.min() > 0 else series.min() * 1.05, series.max() * 1.03)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

def episodes(states, level):
    """Runs of the same regime with start, end, length and the level change over the run."""
    runs, start = [], 0
    s = states.values
    for t in range(1, len(s) + 1):
        if t == len(s) or s[t] != s[start]:
            seg = level.iloc[start:t]
            prev = level.iloc[start - 1] if start > 0 else seg.iloc[0]
            runs.append({"regime": REGIME_NAMES[s[start]], "start": seg.index[0].strftime("%b %Y"),
                         "end": seg.index[-1].strftime("%b %Y"), "months": t - start,
                         "level change": seg.iloc[-1] - prev,
                         "change (%)": 100 * (seg.iloc[-1] / prev - 1) if prev > 0 else np.nan})
            start = t
    return pd.DataFrame(runs)

def episode_summary(ep, label):
    g = ep.groupby("regime")
    out = pd.DataFrame({"episodes": g.size(), "months in regime": g["months"].sum(),
                        "mean episode length (months)": g["months"].mean(), "longest episode (months)": g["months"].max(),
                        "mean change per episode (%)": g["change (%)"].mean()})
    out["method"] = label
    return out.reset_index().set_index(["method", "regime"])

def param_table(info, kind, label):
    rows = []
    for i in range(3):
        row = {"method": label, "state": REGIME_NAMES[i], "pi": info["pi"][i], "a_ii (stay)": info["A"][i, i],
               "to Bear": info["A"][i, 0], "to Stagnant": info["A"][i, 1], "to Bull": info["A"][i, 2]}
        for k, nm in enumerate(SYMBOL_NAMES[kind]):
            row[f"P({nm} | state)"] = info["B"][i, k]
        rows.append(row)
    return pd.DataFrame(rows).set_index(["method", "state"])

handles = [mpatches.Patch(color=c, label=REGIME_NAMES[k]) for k, c in REGIME_COLORS.items()]
wti_chg = changes["WTI_spot"]
wti_lvl = levels["WTI_spot"]

# ---- Alvi's recipe: up/down symbols, three states, no prior, training sequence cut into pieces of 32
t0 = time.time()
wti_states_alvi, wti_model_alvi, wti_info_alvi = detect_regimes(wti_chg, "sign", kappa=0.0, piece=32)
print(f"Alvi's recipe fitted in {time.time() - t0:.1f} s (20 starts x 100 iterations).")

fig, ax = plt.subplots(figsize=(13, 4.2))
regime_plot(ax, wti_lvl, wti_states_alvi, "WTI spot price coloured by regime: Alvi's recipe (up/down symbols, three states, no prior)", "USD per barrel")
shade_events(ax)
ax.axvline(idx_val[0], color=INK, ls=":", lw=1); ax.axvline(idx_test[0], color=INK, ls=":", lw=1)
ax.legend(handles=handles, loc="upper left", ncol=3)
show_fig(fig, 2, "wti_regimes_alvi", "WTI spot price coloured by the decoded regime under Alvi's recipe; dotted lines mark the validation and test blocks, lilac bands the four event windows")

show_table(param_table(wti_info_alvi, "sign", "Alvi"), 2, "wti_hmm_parameters_alvi", "WTI, Alvi's recipe: initial, transition and emission probabilities of the best-of-20 fit on the training block, states named by their mean price change", decimals=3)
ep_alvi = episodes(wti_states_alvi, wti_lvl)
show_table(episode_summary(ep_alvi, "Alvi"), 2, "wti_episode_summary_alvi", "WTI, Alvi's recipe: number, length and price change of the regime episodes", decimals=2)
print(f"Alvi's recipe: {len(ep_alvi)} episodes in {len(wti_states_alvi)} months; the regime changes in {100 * (len(ep_alvi) - 1) / (len(wti_states_alvi) - 1):.0f}% of months.")

### Is it our implementation? Alvi's recipe in `hmmlearn`

The table above shows what the single random start in the dissertation hid: with two symbols and three states, the likelihood is highest for a memoryless solution in which "bear" is simply a down month and "bull" an up month, the stay probabilities are near zero, and almost every episode lasts one month. Before changing the recipe, the first suspect is our own code. Alvi's exact up/down symbols for WTI are therefore fitted with `hmmlearn`'s own Baum-Welch (three states, pieces of 32, 100 iterations), from 20 random starts and from a single start as Alvi did. (Alvi's own library, `hmms`, is tried further below; it has not been maintained since 2018 and does not build on a current Python.)

In [ ]:
obs_alvi = to_symbols(wti_chg.loc[idx_train], "sign")[0].reshape(-1, 1)
lens = [32] * (len(obs_alvi) // 32) + ([len(obs_alvi) % 32] if len(obs_alvi) % 32 else [])
rows = []
for label, seeds in [("hmmlearn, single start (Alvi's practice)", [0]), ("hmmlearn, best of 20 random starts", range(20))]:
    best = None
    for s in seeds:
        h = CategoricalHMM(n_components=3, n_features=2, n_iter=100, random_state=SEED + s, tol=1e-6).fit(obs_alvi, lengths=lens)
        ll = h.score(obs_alvi, lengths=lens)
        if best is None or ll > best[0]:
            best = (ll, h)
    ll, h = best
    order = np.argsort(h.emissionprob_[:, 1])
    A = h.transmat_[np.ix_(order, order)]
    path = h.predict(obs_alvi, lengths=lens)
    rows.append({"fit": label, "log-likelihood": ll, "a_ii Bear": A[0, 0], "a_ii Stagnant": A[1, 1], "a_ii Bull": A[2, 2],
                 "regime switches": int((np.diff(path) != 0).sum()), "months": len(path),
                 "mean episode (months)": len(path) / ((np.diff(path) != 0).sum() + 1)})
rows.append({"fit": "our implementation, best of 20 random starts", "log-likelihood": wti_info_alvi["loglik_best"],
             "a_ii Bear": wti_info_alvi["A"][0, 0], "a_ii Stagnant": wti_info_alvi["A"][1, 1], "a_ii Bull": wti_info_alvi["A"][2, 2],
             "regime switches": int((np.diff(wti_states_alvi.loc[idx_train].values) != 0).sum()), "months": n_train,
             "mean episode (months)": n_train / ((np.diff(wti_states_alvi.loc[idx_train].values) != 0).sum() + 1)})
show_table(pd.DataFrame(rows).set_index("fit"), 2, "alvi_recipe_hmmlearn_check", "Alvi's up/down recipe on the WTI training months fitted by hmmlearn and by our implementation: the regimes flip almost every month under both, so the recipe and not the code is the cause", decimals=3)

### The GWP2 method

The recipe, not the code, produces the degenerate regimes, so the recipe is changed in two places that GWP1 had already promised, keeping Alvi's framework otherwise intact. The alphabet uses the size of the move: the training changes are cut at their quintiles into five symbols, from "large fall" to "large rise", which also centres every series on its own typical move, so a trending series such as CPI is no longer "up" in almost every month. And Baum-Welch runs with a sticky prior of five pseudo-counts on the diagonal of the transition matrix, a mild preference for staying in the current regime. The two methods are compared below on the persistence of the regimes and the episodes they produce; the GWP2 model is the one carried into Steps 4, 6 and 7.

In [ ]:
t0 = time.time()
wti_states_gwp2, wti_model_gwp2, wti_info_gwp2 = detect_regimes(wti_chg, "quintile", kappa=KAPPA_GWP2, n_iter=150)
print(f"GWP2 method fitted in {time.time() - t0:.1f} s (20 starts x up to 150 iterations).")
print("Quintile cut points on the training changes (USD per barrel):", np.round(wti_info_gwp2["thresholds"], 2))

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, st, ttl in [(axes[0], wti_states_alvi, "WTI spot price coloured by regime: Alvi's recipe (up/down symbols, no prior)"),
                    (axes[1], wti_states_gwp2, "WTI spot price coloured by regime: GWP2 method (quintile symbols, sticky prior)")]:
    regime_plot(ax, wti_lvl, st, ttl, "USD per barrel")
    shade_events(ax)
    ax.axvline(idx_val[0], color=INK, ls=":", lw=1); ax.axvline(idx_test[0], color=INK, ls=":", lw=1)
axes[0].legend(handles=handles, loc="upper left", ncol=3)
axes[1].text(idx_val[0], axes[1].get_ylim()[1] * 0.95, " validation", fontsize=8, color=INK)
axes[1].text(idx_test[0], axes[1].get_ylim()[1] * 0.95, " test", fontsize=8, color=INK)
show_fig(fig, 2, "wti_regimes_alvi_vs_gwp2", "WTI spot price coloured by the decoded regime under Alvi's recipe and the GWP2 method; dotted lines mark the validation and test blocks, lilac bands the four event windows")

show_table(param_table(wti_info_gwp2, "quintile", "GWP2"), 2, "wti_hmm_parameters_gwp2", "WTI, GWP2 method: initial, transition and emission probabilities under the quintile alphabet and the sticky prior", decimals=3)

fit_rows = []
for name, info in [("Alvi (up/down, no prior, pieces of 32)", wti_info_alvi), ("GWP2 (quintiles, sticky prior)", wti_info_gwp2)]:
    fit_rows.append({"method": name, "best log-likelihood": info["loglik_best"], "spread across 20 starts": info["loglik_spread"],
                     "share of starts within 1 unit of best": info["starts_within_1"], "Baum-Welch iterations used": len(info["history"])})
show_table(pd.DataFrame(fit_rows).set_index("method"), 2, "wti_fit_stability", "WTI: stability of the Baum-Welch fit across the 20 random starts (log-likelihoods are not comparable across alphabets)", decimals=3)

ep_gwp2 = episodes(wti_states_gwp2, wti_lvl)
ep_cmp = pd.concat([episode_summary(ep_alvi, "Alvi"), episode_summary(ep_gwp2, "GWP2")])
show_table(ep_cmp, 2, "wti_episode_summary", "WTI: number, length and price change of the regime episodes under Alvi's recipe and the GWP2 method", decimals=2)

# Student A: bull episodes; Student B: bear episodes; Student C: stagnant episodes (GWP2 model)
for regime, slug, who in [("Bull", "bull_episodes", "Student A"), ("Bear", "bear_episodes", "Student B"), ("Stagnant", "stagnant_episodes", "Student C")]:
    tbl = ep_gwp2[ep_gwp2["regime"] == regime].sort_values("months", ascending=False).head(8).reset_index(drop=True)
    tbl.index = tbl.index + 1
    show_table(tbl, 2, f"wti_{slug}", f"{who}: the longest {regime.lower()} episodes of the WTI price under the GWP2 model", decimals=2)

# Where the two methods agree
agree = (wti_states_alvi == wti_states_gwp2).mean()
cross = pd.crosstab(wti_states_alvi.map(REGIME_NAMES), wti_states_gwp2.map(REGIME_NAMES), rownames=["Alvi"], colnames=["GWP2"])
show_table(cross, 2, "wti_method_agreement", f"WTI: months by regime under the two methods (they agree on {100 * agree:.1f}% of months)", decimals=None)

## Step 4b: The Regime Process on Every Variable

Alvi's Section 4.2.4 repeats the WTI procedure for every series in the training data: a fresh three-state, two-symbol HMM is drawn at random, trained with 100 Baum-Welch iterations on the training changes cut into pieces of at most 32 months, and the Viterbi path of each block becomes the discrete value of the variable. The Bayesian network is then learned on this table of regime labels. Section 4.2.5 asks for the regime plot of every series to be inspected before the network is trained, because an HMM that has settled on a poor local optimum feeds wrong labels into every later step.

The same is done here for the twenty variables in scope with the GWP2 method of Step 2 (quintile alphabet, sticky prior, ten random starts per series), and three checks per series that the dissertation did not report: the spread of the log-likelihood across starts (a large spread flags a rugged likelihood surface), the persistence $a_{ii}$ of each regime (a regime with $a_{ii}$ below one half flips more often than it stays and is noise rather than a regime), and the share of months in each regime (a state visited in fewer than five percent of months carries almost no information for the network). For comparison, the same checks are run with Alvi's exact method, which shows how many series the up/down alphabet reduces to memoryless labels. Where Alvi's `hmms` package can be installed it is run on the WTI series as a cross-check of our implementation; where it cannot, the report explains the correspondence between its calls and ours.

In [ ]:
N_RESTARTS_PANEL = 10

def discretise_panel(kind, kappa, piece, n_iter):
    states = pd.DataFrame(index=changes.index, dtype=int)
    rows, models = [], {}
    for v in ALL_VARS:
        st, mdl, info = detect_regimes(changes[v], kind, kappa=kappa, n_restarts=N_RESTARTS_PANEL, n_iter=n_iter, piece=piece)
        states[v] = st
        models[v] = (mdl, info)
        shares = st.loc[idx_train].value_counts(normalize=True).reindex([0, 1, 2]).fillna(0)
        up = info["B"][:, N_SYMBOLS[kind] // 2 + (N_SYMBOLS[kind] % 2):].sum(axis=1)     # up: top symbol(s), the "rise" side
        rows.append({"variable": v, "best log-likelihood": info["loglik_best"], "spread across starts": info["loglik_spread"],
                     "starts within 1 unit": info["starts_within_1"],
                     "a_ii Bear": info["A"][0, 0], "a_ii Stagnant": info["A"][1, 1], "a_ii Bull": info["A"][2, 2],
                     "P(rise | Bear)": up[0], "P(rise | Stagnant)": up[1], "P(rise | Bull)": up[2],
                     "share Bear": shares[0], "share Stagnant": shares[1], "share Bull": shares[2]})
    return states, pd.DataFrame(rows).set_index("variable"), models

def flag_table(tbl):
    flags = pd.DataFrame({
        "rugged likelihood (spread > 5)": tbl["spread across starts"] > 5,
        "a regime with a_ii < 0.5": tbl[["a_ii Bear", "a_ii Stagnant", "a_ii Bull"]].min(axis=1) < 0.5,
        "a regime with share < 5%": tbl[["share Bear", "share Stagnant", "share Bull"]].min(axis=1) < 0.05,
    })
    flags["any flag"] = flags.any(axis=1)
    return flags

t0 = time.time()
regime_states, panel_tbl, panel_models = discretise_panel("quintile", KAPPA_GWP2, None, 150)
print(f"{len(ALL_VARS)} series discretised with the GWP2 method in {time.time() - t0:.1f} s ({N_RESTARTS_PANEL} starts x 150 iterations each).")
show_table(panel_tbl, 4, "panel_hmm_stability", "Every variable, GWP2 method: fit stability, regime persistence, probability of a rise symbol in each regime, and the share of training months in each regime", decimals=3)
flags = flag_table(panel_tbl)
show_table(flags, 4, "panel_hmm_flags", "Every variable, GWP2 method: the three warning signs for a regime model that should not be trusted", decimals=None)
print(f"GWP2 method: {int(flags['any flag'].sum())} of {len(flags)} series carry at least one warning sign.")

t0 = time.time()
regime_states_alvi, panel_tbl_alvi, _ = discretise_panel("sign", 0.0, 32, 100)
flags_alvi = flag_table(panel_tbl_alvi)
print(f"Alvi's method on the same series in {time.time() - t0:.1f} s: {int(flags_alvi['any flag'].sum())} of {len(flags_alvi)} series carry a warning sign.")
cmp_methods = pd.DataFrame({
    "GWP2 method": [panel_tbl[["a_ii Bear", "a_ii Stagnant", "a_ii Bull"]].min(axis=1).median(), flags["a regime with a_ii < 0.5"].sum(), flags["a regime with share < 5%"].sum(), flags["any flag"].sum()],
    "Alvi's method": [panel_tbl_alvi[["a_ii Bear", "a_ii Stagnant", "a_ii Bull"]].min(axis=1).median(), flags_alvi["a regime with a_ii < 0.5"].sum(), flags_alvi["a regime with share < 5%"].sum(), flags_alvi["any flag"].sum()],
}, index=["median of the least persistent a_ii", "series with a regime a_ii < 0.5", "series with a regime share < 5%", "series with any flag"])
show_table(cmp_methods, 4, "panel_method_comparison", f"The twenty series under the two regime methods: persistence and warning counts", decimals=3)

# Regime tables for the three blocks, plus the target node WTI_next (the WTI regime one month ahead)
regime_states["WTI_next"] = regime_states["WTI_spot"].shift(-1)
disc_train = regime_states.loc[idx_train].dropna().astype(int)
disc_val = regime_states.loc[idx_val].dropna().astype(int)
disc_test = regime_states.loc[idx_test].dropna().astype(int)
for name, d in [("train", disc_train), ("val", disc_val), ("test", disc_test)]:
    d.to_csv(os.path.join(outdir(4), f"regimes_{name}.csv"))
print(f"Discretised tables saved: train {disc_train.shape}, validation {disc_val.shape}, test {disc_test.shape} (the last training month loses its WTI_next label).")

In [ ]:
ncol = 4
nrow = int(np.ceil(len(ALL_VARS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(18, 3.2 * nrow))
for ax, v in zip(axes.ravel(), ALL_VARS):
    regime_plot(ax, levels[v], regime_states[v], v)
    ax.tick_params(labelsize=8)
for ax in axes.ravel()[len(ALL_VARS):]:
    ax.axis("off")
fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Regime switch plots of every variable (Alvi, Section 4.2.5), GWP2 method: amber bear, grey stagnant, purple bull", y=1.0)
fig.tight_layout()
show_fig(fig, 4, "panel_regime_plots", "Regime switch plots of the twenty variables under the GWP2 method, decoded with the training-block parameters")

# Co-movement of the regime labels on the training block (Cramer's V between pairs of regime series)
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0.0
V = pd.DataFrame(index=ALL_VARS, columns=ALL_VARS, dtype=float)
for a in ALL_VARS:
    for b in ALL_VARS:
        V.loc[a, b] = 1.0 if a == b else cramers_v(disc_train[a], disc_train[b])
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(V.astype(float), cmap=PURPLE_CMAP, vmin=0, vmax=1, square=True, cbar_kws={"shrink": 0.7}, ax=ax, linewidths=0.3, linecolor="white")
ax.set_title("Cramer's V between the regime labels of every pair of variables (training block)")
show_fig(fig, 4, "regime_cramers_v", "Association between the regime labels of the twenty variables on the training block (Cramer's V)")
top_pairs = V.where(np.triu(np.ones(V.shape), 1).astype(bool)).stack().sort_values(ascending=False).head(12)
top_pairs = top_pairs.rename("Cramer's V").to_frame()
show_table(top_pairs, 4, "regime_top_associations", "The twelve most associated pairs of regime series on the training block", decimals=3)

### Alvi's `hmms` package, if it installs

Alvi used `hmms` (Lopatovsky, 2017), a Cython library whose `DtHMM.random(3, 2)`, `baum_welch(sequences, 100)`, `viterbi(sequence)` and `print_parameters(model)` calls map one to one onto `DiscreteHMM(3, 2)`, `baum_welch(pieces, 100)`, `viterbi(symbols)` and the parameter table above. The package has not been maintained since 2018 and may not build on a current Python. The cell below tries to install it and, if it succeeds, fits the WTI training sequence with it and compares the log-likelihood with ours; if it fails, the notebook records that and moves on.

In [ ]:
HMMS_OK = False
try:
    import hmms
    HMMS_OK = True
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hmms"], capture_output=True, text=True)
    try:
        import hmms
        HMMS_OK = True
    except Exception as exc:
        print("hmms could not be installed on this Python:", str(exc)[:200])
if HMMS_OK:
    try:
        sym_tr, _ = to_symbols(wti_chg.loc[idx_train], "sign")
        pieces = np.array_split(sym_tr, int(np.ceil(len(sym_tr) / 32)))
        np.random.seed(SEED)
        dhmm = hmms.DtHMM.random(3, 2)
        dhmm.baum_welch(pieces, 100)
        hmms.print_parameters(dhmm)
        ll_hmms = sum(dhmm.data_estimate(p) for p in pieces)
        print(f"hmms pooled log-likelihood after 100 iterations from one random start: {ll_hmms:.3f}")
        print(f"our best-of-20 pooled log-likelihood on the same pieces (Alvi's method): {wti_info_alvi['loglik_best']:.3f}")
    except Exception as exc:
        print("hmms imported but the call sequence failed:", str(exc)[:300])
else:
    print("Proceeding with the from-scratch implementation; the report explains the correspondence of the calls.")

## Step 9: A Minimal Working Example of Hill Climbing

Alvi's Section 4.2.6 learns the structure of the oil market network with `pgmpy`'s `HillClimbSearch`: starting from a small expert graph of EIA knowledge (supply and demand to the price, GDP to demand), the search adds, deletes or reverses one arc at a time and keeps the move that most improves a decomposable score, until no move improves it. He used the K2 score; the BIC and BDeu scores are the two other standard choices and are run alongside. The example below works on the Alvi-like variable set, with six expert arcs mirroring his:

- `US_crude_prod` and `US_crude_imports` to `WTI_spot` (supply to price);
- `Refinery_util` and `Gasoline_demand` to `WTI_spot` (demand to price);
- `RealGDP` to `Gasoline_demand` and `INDPRO` to `Refinery_util` (activity to demand);
- `WTI_spot` to `WTI_next` (the current regime to next month's, the arc that makes the forecast node reachable).

The learned graph is then fitted with Bayesian parameter estimation under a K2 prior, as in the dissertation, and the arcs the search added or removed are listed.

In [ ]:
try:
    from pgmpy.models import DiscreteBayesianNetwork as BN
except ImportError:
    from pgmpy.models import BayesianNetwork as BN
from pgmpy.base import DAG
from pgmpy.estimators import HillClimbSearch, BayesianEstimator, MaximumLikelihoodEstimator
try:
    from pgmpy.estimators import BIC, K2, BDeu
except ImportError:
    from pgmpy.estimators import BicScore as BIC, K2Score as K2, BDeuScore as BDeu
from pgmpy.inference import VariableElimination
import logging
logging.getLogger("pgmpy").setLevel(logging.ERROR)

# Reproducibility: pgmpy picks the best move with max() over a set of candidate moves, and under the score-equivalent
# BIC and BDeu scores "add X -> Y" and "add Y -> X" tie exactly, so the winner depends on Python's per-process hash order.
# Sorting the candidates by rounded score and then by name makes the tie-break explicit and every run identical.
if hasattr(HillClimbSearch, "_legal_operations"):
    _orig_legal_operations = HillClimbSearch._legal_operations
    def _sorted_legal_operations(self, *args, **kwargs):
        ops = list(_orig_legal_operations(self, *args, **kwargs))
        ops.sort(key=lambda t: (-round(float(t[1]), 9), str(t[0])))
        yield from ops
    HillClimbSearch._legal_operations = _sorted_legal_operations

SCORES = {"K2": K2, "BIC": BIC, "BDeu": BDeu}
STATE_NAMES = lambda cols: {c: [0, 1, 2] for c in cols}

def make_dag(nodes, arcs):
    g = DAG()
    g.add_nodes_from(nodes)
    g.add_edges_from(arcs)
    return g

try:
    from pgmpy.causal_discovery import ExpertKnowledge
except ImportError:
    try:
        from pgmpy.estimators import ExpertKnowledge
    except ImportError:
        ExpertKnowledge = None

def hill_climb(data, start_arcs, score_name, max_indegree=4, target="WTI_next"):
    """Hill climbing from an expert graph; returns the learned DAG and the scores before and after.
    Arcs out of the target (next month's regime) are forbidden: a future value cannot be a parent of this month's variables."""
    cols = list(data.columns)
    scorer = SCORES[score_name](data)
    start = make_dag(cols, start_arcs)
    hc = HillClimbSearch(data)
    forbidden = [(target, c) for c in cols if c != target]
    np.random.seed(SEED); random.seed(SEED)              # pgmpy's search reads the global RNG; pin it so every run learns the same graph
    common = dict(scoring_method=scorer, start_dag=start, max_indegree=max_indegree, show_progress=False)
    learned = None
    if ExpertKnowledge is not None:
        try:
            learned = hc.estimate(expert_knowledge=ExpertKnowledge(forbidden_edges=forbidden), **common)
        except TypeError:
            learned = None
    if learned is None:                                   # older pgmpy releases take a black list instead
        learned = hc.estimate(black_list=forbidden, **common)
    return learned, scorer.score(start), scorer.score(learned)

def arc_changes(start_arcs, learned):
    s = set(start_arcs)
    l = set(learned.edges())
    kept = sorted(s & l)
    reversed_ = sorted(e for e in s if (e[1], e[0]) in l)
    dropped = sorted(e for e in s - l if (e[1], e[0]) not in l)
    added = sorted(e for e in l - s if (e[1], e[0]) not in s)
    return kept, reversed_, dropped, added

def fit_bn(learned, data, estimator="K2"):
    m = BN()
    m.add_nodes_from(list(data.columns))
    m.add_edges_from(learned.edges())
    # The estimators are called directly (model.fit changed its keyword interface between pgmpy releases).
    if estimator == "MLE":
        cpds = MaximumLikelihoodEstimator(m, data, state_names=STATE_NAMES(data.columns)).get_parameters()
    else:
        cpds = BayesianEstimator(m, data, state_names=STATE_NAMES(data.columns)).get_parameters(prior_type="K2")
    m.add_cpds(*cpds)
    assert m.check_model()
    return m

ALVI_COLS = ALVI_LIKE + ["WTI_next"]
alvi_train = disc_train[ALVI_COLS]
EXPERT_ALVI = [("US_crude_prod", "WTI_spot"), ("US_crude_imports", "WTI_spot"), ("Refinery_util", "WTI_spot"),
               ("Gasoline_demand", "WTI_spot"), ("RealGDP", "Gasoline_demand"), ("INDPRO", "Refinery_util"),
               ("WTI_spot", "WTI_next")]

mwe_rows, mwe_models = [], {}
t0 = time.time()
for sc in SCORES:
    learned, s0, s1 = hill_climb(alvi_train, EXPERT_ALVI, sc)
    kept, rev, drop, add = arc_changes(EXPERT_ALVI, learned)
    mwe_models[sc] = learned
    mwe_rows.append({"score": sc, "score of expert graph": s0, "score of learned graph": s1, "arcs in learned graph": len(learned.edges()),
                     "expert arcs kept": len(kept), "expert arcs reversed": len(rev), "expert arcs dropped": len(drop), "arcs added": len(add),
                     "parents of WTI_next": ", ".join(sorted(learned.get_parents("WTI_next"))) or "(none)"})
print(f"Three hill climbs on {alvi_train.shape[0]} training months x {alvi_train.shape[1]} regime variables in {time.time() - t0:.1f} s.")
mwe_tbl = pd.DataFrame(mwe_rows).set_index("score")
show_table(mwe_tbl, 9, "hill_climb_minimal_example", "Minimal working example: hill climbing from the six expert arcs (plus the forecast arc) on the Alvi-like variable set under three scores", decimals=1)

kept, rev, drop, add = arc_changes(EXPERT_ALVI, mwe_models["K2"])
moves = pd.DataFrame({"move": ["kept"] * len(kept) + ["reversed"] * len(rev) + ["dropped"] * len(drop) + ["added"] * len(add),
                      "arc": [f"{a} -> {b}" for a, b in kept + rev + drop + add]})
show_table(moves.set_index("move"), 9, "hill_climb_k2_moves", "Minimal working example: what the K2 hill climb did with each expert arc and which arcs it added", decimals=None)

def draw_dag(learned, title, highlight=(), seed=SEED):
    G = nx.DiGraph(); G.add_nodes_from(learned.nodes()); G.add_edges_from(learned.edges())
    pos = nx.spring_layout(G, seed=seed, k=1.6)
    fig, ax = plt.subplots(figsize=(11, 7.5))
    colors = [AMBER if n in ("WTI_spot", "WTI_next") else WQU_PURPLE for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1500, alpha=0.95, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_color="white", font_weight="bold", ax=ax)
    hl = set(highlight)
    nx.draw_networkx_edges(G, pos, edgelist=[e for e in G.edges() if e in hl], edge_color=AMBER, width=2.2, arrowsize=18, ax=ax, connectionstyle="arc3,rad=0.08")
    nx.draw_networkx_edges(G, pos, edgelist=[e for e in G.edges() if e not in hl], edge_color=GREY, width=1.3, arrowsize=16, ax=ax, connectionstyle="arc3,rad=0.08")
    ax.set_title(title); ax.axis("off")
    return fig

fig = draw_dag(mwe_models["K2"], "Minimal working example: the K2 hill climb from the expert graph (amber arcs are the expert arcs that survived)", highlight=kept)
show_fig(fig, 9, "hill_climb_minimal_example_dag", "The network learned by K2 hill climbing on the Alvi-like variable set; expert arcs that survived are amber")

mwe_bn = fit_bn(mwe_models["K2"], alvi_train)
cpd = mwe_bn.get_cpds("WTI_next")
print(f"WTI_next in the minimal example has parents {cpd.variables[1:]} and a CPD with {int(np.prod(cpd.cardinality[1:]))} parent configurations.")
print(cpd)

## Step 4c: Training the Network with `pgmpy`, and Testing and Validating the Parameters

The full run uses the GWP1 variable set and starts the hill climb from the 23 arcs of the GWP1 Inferred Causality graph (Table 22 of the GWP1 report), with `WTI_spot` to `WTI_next` added so that the forecast node is reachable. Three questions are answered in turn:

1. **Structure.** Which arcs of the constraint-based GWP1 graph survive a score-based search on regime labels, which are reversed, which are dropped, and which arcs appear that the constraint-based search had not found. The three scores are compared and the learned graphs drawn.
2. **Parameters.** The conditional probability tables are estimated by maximum likelihood and by Bayesian estimation with a K2 prior (Alvi's choice, which adds one pseudo-count to every cell and keeps zero-count cells away from zero). The table for the forecast node is shown both ways, and the empirical transition between this month's and next month's WTI regime is given for comparison.
3. **Validation and test.** Alvi's Section 4.3 discretises the validation block with the trained HMMs, feeds the regimes as evidence and compares the predicted regime of the forecast node with the realised one. The same is done here with variable elimination: the most probable `WTI_next` regime given all other regimes of the month, scored by accuracy, by the log-loss of the predicted distribution, and by a confusion matrix, against two baselines, persistence (next month's regime equals this month's, with the training transition table as its distribution) and the majority regime of the training block. Because regimes persist, the current WTI regime is the strongest single piece of evidence; a second query withholds it and asks what the other drivers alone say about next month, which is the information the network adds beyond persistence. The score whose network does best on the validation block (ties broken towards the sparser graph) is the one carried to the test block, which is evaluated once.
4. **Robustness.** The training months are resampled with replacement fifty times (seeded) and the hill climb repeated; the share of resamples in which each arc appears says how much of the structure is supported by the data rather than by one sample.

In [ ]:
GWP1_ARCS = [("CPI", "INDPRO"), ("CPI", "UST10Y"), ("CPI", "WTI_spot"), ("Drilling_idx", "INDPRO"), ("Drilling_idx", "OVX"),
             ("Drilling_idx", "US_crude_stocks"), ("FedFunds", "CPI"), ("FedFunds", "OVX"), ("GEPU", "OVX"), ("GPR", "OVX"),
             ("Gasoline_demand", "INDPRO"), ("Gasoline_demand", "Refinery_util"), ("Gold_fut", "DXY"), ("INDPRO", "OVX"),
             ("Refinery_util", "Gold_fut"), ("Refinery_util", "INDPRO"), ("Refinery_util", "US_crude_stocks"), ("SPX", "DXY"),
             ("SPX", "WTI_spot"), ("UST10Y", "Gold_fut"), ("UST10Y", "WTI_spot"), ("US_crude_prod", "Refinery_util"), ("WTI_spot", "OVX")]
START_FULL = GWP1_ARCS + [("WTI_spot", "WTI_next")]
FULL_COLS = GWP1_NODES + ["WTI_next"]
full_train, full_val, full_test = disc_train[FULL_COLS], disc_val[FULL_COLS], disc_test[FULL_COLS]

full_rows, full_models = [], {}
t0 = time.time()
for sc in SCORES:
    learned, s0, s1 = hill_climb(full_train, START_FULL, sc)
    kept, rev, drop, add = arc_changes(GWP1_ARCS, learned)
    full_models[sc] = learned
    full_rows.append({"score": sc, "score of start graph": s0, "score of learned graph": s1, "arcs in learned graph": len(learned.edges()),
                      "GWP1 arcs kept": len(kept), "GWP1 arcs reversed": len(rev), "GWP1 arcs dropped": len(drop), "arcs added": len(add),
                      "parents of WTI_next": ", ".join(sorted(learned.get_parents("WTI_next"))) or "(none)",
                      "Markov blanket of WTI_next": ", ".join(sorted(learned.get_markov_blanket("WTI_next"))) or "(none)"})
print(f"Three hill climbs on {full_train.shape[0]} training months x {full_train.shape[1]} regime variables in {time.time() - t0:.1f} s.")
full_tbl = pd.DataFrame(full_rows).set_index("score")
show_table(full_tbl, 4, "hill_climb_full_summary", "Full run: hill climbing from the 23 GWP1 arcs on the sixteen GWP1 variables plus the forecast node, under three scores", decimals=1)

arc_rows = []
for sc, learned in full_models.items():
    kept, rev, drop, add = arc_changes(GWP1_ARCS, learned)
    for lab, lst in [("kept", kept), ("reversed", rev), ("dropped", drop), ("added", add)]:
        for a, b in lst:
            arc_rows.append({"score": sc, "move": lab, "arc": f"{a} -> {b}"})
arc_tbl = pd.DataFrame(arc_rows)
arc_pivot = arc_tbl.pivot_table(index="arc", columns="score", values="move", aggfunc="first").fillna("")
arc_pivot["in GWP1 graph"] = [any(f"{a} -> {b}" == i for a, b in GWP1_ARCS) for i in arc_pivot.index]
show_table(arc_pivot, 4, "hill_climb_full_arcs", "Full run: the fate of every GWP1 arc and every added arc under the three scores", decimals=None)

for sc in SCORES:
    kept = arc_changes(GWP1_ARCS, full_models[sc])[0]
    fig = draw_dag(full_models[sc], f"Full run: network learned by {sc} hill climbing from the GWP1 graph (amber arcs kept from GWP1)", highlight=kept)
    show_fig(fig, 4, f"hill_climb_full_dag_{sc.lower()}", f"The network learned by {sc} hill climbing on the GWP1 variables; arcs kept from the GWP1 Inferred Causality graph are amber")

In [ ]:
# ---- Parameters: the forecast node's CPD by MLE and by Bayesian estimation with a K2 prior
def cpd_frame(bn, node):
    cpd = bn.get_cpds(node)
    parents = cpd.variables[1:]
    vals = cpd.get_values()
    if parents:
        cols = pd.MultiIndex.from_tuples(list(itertools.product(*[[REGIME_NAMES[s] for s in cpd.state_names[p]] for p in parents])), names=parents)
    else:
        cols = ["marginal"]
    return pd.DataFrame(vals, index=[f"P({node} = {REGIME_NAMES[s]})" for s in cpd.state_names[node]], columns=cols)

best_sc_struct = "K2"
bn_k2 = fit_bn(full_models[best_sc_struct], full_train, "K2")
bn_mle = fit_bn(full_models[best_sc_struct], full_train, "MLE")
cpd_bayes = cpd_frame(bn_k2, "WTI_next")
cpd_mle = cpd_frame(bn_mle, "WTI_next")
print(f"WTI_next under the {best_sc_struct} structure: parents {list(cpd_bayes.columns.names) if cpd_bayes.columns.nlevels > 1 or cpd_bayes.columns.names[0] else '(none)'}, {cpd_bayes.shape[1]} parent configurations.")
show_table(cpd_bayes.T.head(27), 4, "wti_next_cpd_bayes_k2", "Full run: conditional probability table of WTI_next by Bayesian estimation with a K2 prior (first 27 parent configurations)", decimals=3)
show_table(cpd_mle.T.head(27), 4, "wti_next_cpd_mle", "Full run: the same table by maximum likelihood; empty parent configurations give undefined rows that the K2 prior fills with equal probabilities", decimals=3)

counts = pd.crosstab(full_train["WTI_spot"].map(REGIME_NAMES), full_train["WTI_next"].map(REGIME_NAMES), rownames=["WTI regime this month"], colnames=["WTI regime next month"])
show_table(counts, 4, "wti_regime_transition_counts", "Training block: counts of this month's against next month's WTI regime (the empirical transition table behind the forecast)", decimals=None)
show_table(counts.div(counts.sum(axis=1), axis=0), 4, "wti_regime_transition_probs", "Training block: the empirical transition probabilities of the WTI regime", decimals=3)
n_params_k2 = sum(int(np.prod(c.cardinality[1:])) * (c.cardinality[0] - 1) for c in bn_k2.get_cpds())
print(f"The {best_sc_struct} network has {len(bn_k2.edges())} arcs and {n_params_k2} free parameters, learned from {full_train.shape[0]} training months.")

In [ ]:
# ---- Validation and test: predict next month's WTI regime from this month's regimes of the other variables
def predict_block(bn, block, withhold=()):
    """Posterior of WTI_next for every month of the block given the other regimes of the month (minus any withheld)."""
    infer = VariableElimination(bn)
    evid_cols = [c for c in block.columns if c != "WTI_next" and c not in withhold]
    probs = []
    for _, row in block.iterrows():
        ev = {c: int(row[c]) for c in evid_cols if c in bn.nodes()}
        q = infer.query(variables=["WTI_next"], evidence=ev, show_progress=False)
        probs.append(np.asarray(q.values, dtype=float))
    return np.vstack(probs)

def score_block(name, y_true, probs):
    y_pred = probs.argmax(axis=1)
    p_true = np.clip(probs[np.arange(len(y_true)), y_true], 1e-6, 1)
    return {"block": name, "accuracy": (y_true == y_pred).mean(), "log-loss": -np.log(p_true).mean(),
            "bear recall": ((y_pred == 0) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1),
            "bull recall": ((y_pred == 2) & (y_true == 2)).sum() / max((y_true == 2).sum(), 1)}

# Baselines as distributions: persistence uses the training transition table of the WTI regime, majority a point mass
trans = pd.crosstab(full_train["WTI_spot"], full_train["WTI_next"]).reindex(index=[0, 1, 2], columns=[0, 1, 2]).fillna(0) + 0.5
trans = trans.div(trans.sum(axis=1), axis=0).values
majority = int(full_train["WTI_next"].mode()[0])
def persistence_probs(block):
    return trans[block["WTI_spot"].values]
def majority_probs(block):
    p = np.full((len(block), 3), 1e-6); p[:, majority] = 1 - 2e-6; return p

val_rows, val_probs, val_probs_drivers = [], {}, {}
t0 = time.time()
y_val = full_val["WTI_next"].values
for sc in SCORES:
    bn = fit_bn(full_models[sc], full_train, "K2")
    val_probs[sc] = predict_block(bn, full_val)
    val_probs_drivers[sc] = predict_block(bn, full_val, withhold=("WTI_spot",))
    val_rows.append({"model": f"hill climb {sc} + K2 prior, all evidence", "arcs": len(bn.edges()), **score_block("validation", y_val, val_probs[sc])})
    val_rows.append({"model": f"hill climb {sc} + K2 prior, drivers only (WTI regime withheld)", "arcs": len(bn.edges()), **score_block("validation", y_val, val_probs_drivers[sc])})
val_rows.append({"model": "persistence (training transition table)", "arcs": 0, **score_block("validation", y_val, persistence_probs(full_val))})
val_rows.append({"model": f"majority regime of training ({REGIME_NAMES[majority]})", "arcs": 0, **score_block("validation", y_val, majority_probs(full_val))})
print(f"Validation inference in {time.time() - t0:.1f} s.")
val_tbl = pd.DataFrame(val_rows).set_index("model")
show_table(val_tbl, 4, "validation_scores", f"Validation block ({len(full_val)} months): accuracy and log-loss of the predicted WTI regime one month ahead, with and without this month's WTI regime as evidence, against two baselines", decimals=3)

cand = val_tbl.loc[[f"hill climb {sc} + K2 prior, all evidence" for sc in SCORES]].copy()
cand["score"] = list(SCORES)
best_sc = cand.sort_values(["accuracy", "log-loss", "arcs"], ascending=[False, True, True]).iloc[0]["score"]
print(f"Carried to the test block: hill climb {best_sc} with the K2 prior (highest validation accuracy, then lowest log-loss, then fewest arcs).")
cm_val = pd.crosstab(pd.Series(y_val).map(REGIME_NAMES), pd.Series(val_probs[best_sc].argmax(axis=1)).map(REGIME_NAMES), rownames=["realised"], colnames=["predicted"])
show_table(cm_val, 4, "validation_confusion", f"Validation block: confusion matrix of the {best_sc} network with all evidence", decimals=None)

bn_best = fit_bn(full_models[best_sc], full_train, "K2")
y_test = full_test["WTI_next"].values
p_test = predict_block(bn_best, full_test)
p_test_drivers = predict_block(bn_best, full_test, withhold=("WTI_spot",))
test_rows = [{"model": f"hill climb {best_sc} + K2 prior, all evidence", **score_block("test", y_test, p_test)},
             {"model": f"hill climb {best_sc} + K2 prior, drivers only (WTI regime withheld)", **score_block("test", y_test, p_test_drivers)},
             {"model": "persistence (training transition table)", **score_block("test", y_test, persistence_probs(full_test))},
             {"model": f"majority regime of training ({REGIME_NAMES[majority]})", **score_block("test", y_test, majority_probs(full_test))}]
show_table(pd.DataFrame(test_rows).set_index("model"), 4, "test_scores", f"Test block ({len(full_test)} months), evaluated once: the {best_sc} network against the two baselines", decimals=3)
cm_test = pd.crosstab(pd.Series(y_test).map(REGIME_NAMES), pd.Series(p_test.argmax(axis=1)).map(REGIME_NAMES), rownames=["realised"], colnames=["predicted"])
show_table(cm_test, 4, "test_confusion", f"Test block: confusion matrix of the {best_sc} network with all evidence", decimals=None)

# Months where the regime changes: does any model see the turn coming?
turn_val = y_val != full_val["WTI_spot"].values
turn_test = y_test != full_test["WTI_spot"].values
turns = pd.DataFrame({
    "regime changes in block": [int(turn_val.sum()), int(turn_test.sum())],
    "network, all evidence": [(val_probs[best_sc].argmax(axis=1)[turn_val] == y_val[turn_val]).mean() if turn_val.any() else np.nan,
                              (p_test.argmax(axis=1)[turn_test] == y_test[turn_test]).mean() if turn_test.any() else np.nan],
    "network, drivers only": [(val_probs_drivers[best_sc].argmax(axis=1)[turn_val] == y_val[turn_val]).mean() if turn_val.any() else np.nan,
                              (p_test_drivers.argmax(axis=1)[turn_test] == y_test[turn_test]).mean() if turn_test.any() else np.nan],
    "persistence": [0.0, 0.0],
}, index=["validation", "test"])
show_table(turns, 4, "turning_point_accuracy", f"Accuracy on the months where the WTI regime changes (persistence is wrong on every one of them by construction)", decimals=3)

# Month by month on the validation and test blocks
timeline = pd.DataFrame({"realised": np.concatenate([y_val, y_test]),
                         "predicted": np.concatenate([val_probs[best_sc].argmax(axis=1), p_test.argmax(axis=1)])},
                        index=list(full_val.index) + list(full_test.index))
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.step(timeline.index, timeline["realised"], where="mid", color=INK, lw=1.5, label="realised WTI regime next month")
ax.step(timeline.index, timeline["predicted"], where="mid", color=AMBER, lw=1.5, ls="--", label=f"predicted ({best_sc} network)")
ax.axvline(idx_test[0], color=INK, ls=":", lw=1)
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Bear", "Stagnant", "Bull"]); ax.legend(loc="upper left")
ax.set_title("Validation and test blocks: realised against predicted WTI regime one month ahead")
show_fig(fig, 4, "validation_test_timeline", f"Realised and predicted WTI regime one month ahead on the validation and test blocks ({best_sc} network; dotted line starts the test block)")

In [ ]:
# ---- Robustness: bootstrap the training months and count how often each arc is learned
N_BOOT = 50
t0 = time.time()
arc_counts = {}
boot_rng = np.random.default_rng(SEED)
for b in range(N_BOOT):
    sample = full_train.iloc[boot_rng.integers(0, len(full_train), len(full_train))].reset_index(drop=True)
    learned, _, _ = hill_climb(sample, START_FULL, best_sc, max_indegree=3)
    for e in learned.edges():
        arc_counts[e] = arc_counts.get(e, 0) + 1
print(f"{N_BOOT} bootstrap hill climbs ({best_sc}, max in-degree 3) in {time.time() - t0:.1f} s.")
boot = pd.DataFrame([{"arc": f"{a} -> {b}", "share of resamples": c / N_BOOT,
                      "in GWP1 graph": (a, b) in GWP1_ARCS, f"in {best_sc} graph": (a, b) in set(full_models[best_sc].edges())}
                     for (a, b), c in arc_counts.items()]).sort_values("share of resamples", ascending=False).set_index("arc")
show_table(boot.head(30), 4, "bootstrap_arc_frequency", f"The thirty arcs most often learned across {N_BOOT} bootstrap resamples of the training months", decimals=2)
stable = boot[boot["share of resamples"] >= 0.5]
print(f"{len(stable)} arcs appear in at least half of the resamples; {int(stable['in GWP1 graph'].sum())} of them are GWP1 arcs.")

fig, ax = plt.subplots(figsize=(9, 7))
top = boot.head(25).iloc[::-1]
ax.barh(top.index, top["share of resamples"], color=[AMBER if g else WQU_PURPLE for g in top["in GWP1 graph"]])
ax.axvline(0.5, color=INK, ls=":", lw=1); ax.set_xlim(0, 1); ax.set_xlabel("share of bootstrap resamples in which the arc is learned")
ax.set_title(f"Arc stability over {N_BOOT} bootstrap hill climbs (amber: arcs of the GWP1 graph)")
show_fig(fig, 4, "bootstrap_arc_stability", f"Bootstrap stability of the learned arcs ({best_sc} score); amber bars are arcs already in the GWP1 Inferred Causality graph")

## Steps 6 and 7: How the Regimes Are Learned, Decoded and Interpreted

Section 4.2 of the dissertation is the source for the three Step 6 write-ups. The material each student needs is collected here from the WTI model of Step 2:

- **Transforming the time series (Student A).** The level series becomes a sequence of monthly changes, the changes become symbols (Alvi's up/down, or the GWP2 quintiles cut at the training block's quantiles), and Alvi's library then cuts the symbol sequence into pieces of at most 32 for training. The table below records how many months each transformation keeps and how the symbols are distributed in each block.
- **How the parameters are learned (Student B).** The Baum-Welch trace of the best random start, the spread of final log-likelihoods across starts, and the learned $\Pi$, $A$ and $B$ (Step 2) are what the write-up describes, treating the algorithm as a black box that climbs the likelihood.
- **The most likely sequence of hidden states (Student C).** The Viterbi path over the whole sample, compared with the month-by-month posterior argmax from the forward and backward passes, shows where the two decoders differ (Viterbi enforces a coherent path, the posterior does not).

**Step 7, the latent meaning of each hidden state**, is read off the decoded regimes: the mean and spread of the monthly change in each state, the probability of an up month, the expected duration $1 / (1 - a_{ii})$, and the historical episodes that fall in each state.

In [ ]:
# Student A: the transformation chain
cuts = wti_info_gwp2["thresholds"]
sym_full, _ = to_symbols(wti_chg, "quintile", cuts)
sym_alvi, _ = to_symbols(wti_chg, "sign")
chain = pd.DataFrame({
    "stage": ["monthly levels on the common sample", "monthly changes (first month lost)", "symbols (one per change)",
              "training months", "training pieces of at most 32 symbols (Alvi's library)"],
    "count": [len(levels) + 1, len(changes), len(sym_full), n_train, int(np.ceil(n_train / 32))],
}).set_index("stage")
show_table(chain, 6, "transformation_chain", "WTI: what each transformation step keeps, from levels to the training pieces Alvi's Baum-Welch consumed", decimals=None)
blocks = {"training": slice(0, n_train), "validation": slice(n_train, n_train + n_val), "test": slice(n_train + n_val, None)}
sym_mix = pd.DataFrame({"months": [n_train, n_val, n_test],
                        "share of up months (Alvi symbols)": [sym_alvi[s].mean() for s in blocks.values()],
                        **{f"share {nm}": [(sym_full[s] == k).mean() for s in blocks.values()] for k, nm in enumerate(SYMBOL_NAMES["quintile"])}},
                       index=list(blocks))
show_table(sym_mix, 6, "symbol_mix_by_block", f"WTI: symbol mix in each block; the quintile cuts {np.round(cuts, 2)} USD come from the training block, so later blocks need not be evenly spread", decimals=3)

# Student B: the learning trace
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(wti_info_gwp2["history"], color=WQU_PURPLE, label="GWP2 method (quintiles, sticky prior)")
ax2 = ax.twinx(); ax2.plot(wti_info_alvi["history"], color=AMBER, label="Alvi's method (up/down, pieces of 32)"); ax2.set_ylabel("log-likelihood, Alvi", color=AMBER)
ax.set_xlabel("Baum-Welch iteration"); ax.set_ylabel("log-likelihood, GWP2", color=WQU_PURPLE)
ax.set_title("WTI: Baum-Welch trace of the best of 20 random starts under each method")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="lower right")
show_fig(fig, 6, "wti_baum_welch_trace", "WTI: log-likelihood per Baum-Welch iteration for the best random start of each method")

# Student C: Viterbi against the posterior argmax (GWP2 model)
mapping_gwp2 = label_by_mean(wti_model_gwp2.viterbi(to_symbols(wti_chg.loc[idx_train], "quintile", cuts)[0])[0], wti_chg.loc[idx_train].values)
post_full, _ = wti_model_gwp2.posterior(sym_full)
post_argmax = pd.Series([mapping_gwp2[s] for s in post_full.argmax(axis=1)], index=changes.index)
vit_full = pd.Series([mapping_gwp2[s] for s in wti_model_gwp2.viterbi(sym_full)[0]], index=changes.index)
diff_months = (vit_full != post_argmax).sum()
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
regime_plot(axes[0], wti_lvl, vit_full, "WTI: Viterbi path decoded over the whole sample with the training parameters (GWP2 model)", "USD per barrel")
shade_events(axes[0])
perm_gwp2 = sorted(mapping_gwp2, key=mapping_gwp2.get)          # raw state ids in bear, stagnant, bull order
axes[1].stackplot(changes.index, post_full[:, perm_gwp2].T, colors=[AMBER, GREY, WQU_PURPLE], alpha=0.85)
axes[1].set_ylim(0, 1); axes[1].set_title(f"Posterior regime probabilities (forward and backward); the argmax differs from Viterbi in {diff_months} months")
axes[0].legend(handles=handles, loc="upper left", ncol=3)
show_fig(fig, 6, "wti_viterbi_vs_posterior", f"WTI: the Viterbi path and the month-by-month posterior probabilities; the two decoders disagree in {diff_months} of {len(changes)} months")

In [ ]:
# Step 7: latent meaning of each hidden state (WTI, GWP2 model, whole sample decoded with training parameters)
A_s = wti_info_gwp2["A"]; B_s = wti_info_gwp2["B"]
rows = []
for k in range(3):
    mask = wti_states_gwp2 == k
    ep_k = ep_gwp2[ep_gwp2["regime"] == REGIME_NAMES[k]]
    rows.append({"state": REGIME_NAMES[k],
                 "months": int(mask.sum()), "share of months": mask.mean(),
                 "mean monthly change (USD)": wti_chg[mask].mean(), "sd of monthly change (USD)": wti_chg[mask].std(),
                 "share of up months": (wti_chg[mask] > 0).mean(), "P(rise or large rise | state) from B": B_s[k, 3:].sum(),
                 "P(fall or large fall | state) from B": B_s[k, :2].sum(),
                 "a_ii (stay)": A_s[k, k], "expected duration 1/(1-a_ii) (months)": 1 / (1 - A_s[k, k]) if A_s[k, k] < 1 else np.inf,
                 "observed mean episode (months)": ep_k["months"].mean(), "episodes": len(ep_k),
                 "longest episode": f"{ep_k.sort_values('months').iloc[-1]['start']} to {ep_k.sort_values('months').iloc[-1]['end']}" if len(ep_k) else ""})
latent = pd.DataFrame(rows).set_index("state")
show_table(latent, 7, "latent_meaning_wti", "Step 7: what each hidden state of the WTI model means, from the decoded regimes and the learned parameters", decimals=3)

# The same reading for every variable: mean change by state, so the report can name the states of the drivers
lat_rows = []
for v in ALL_VARS:
    mdl, info = panel_models[v]
    for k in range(3):
        mask = regime_states[v] == k
        lat_rows.append({"variable": v, "state": REGIME_NAMES[k], "share of months": mask.mean(),
                         "mean change": changes[v][mask].mean(), "share up": (changes[v][mask] > 0).mean(), "a_ii": info["A"][k, k]})
lat_all = pd.DataFrame(lat_rows).set_index(["variable", "state"])
show_table(lat_all, 7, "latent_meaning_all_variables", "Step 7: share of months, mean change, share of up months and persistence of each hidden state of every variable", decimals=3)

# Which regimes of the drivers coincide with a bear WTI month next month, on the training block
driver_rows = []
for v in [c for c in FULL_COLS if c not in ("WTI_next",)]:
    ct = pd.crosstab(full_train[v], full_train["WTI_next"], normalize="index").reindex(index=[0, 1, 2], columns=[0, 1, 2]).fillna(0)
    driver_rows.append({"variable": v, "P(WTI bear next | var Bear)": ct.loc[0, 0], "P(WTI bear next | var Bull)": ct.loc[2, 0],
                        "P(WTI bull next | var Bear)": ct.loc[0, 2], "P(WTI bull next | var Bull)": ct.loc[2, 2]})
drivers = pd.DataFrame(driver_rows).set_index("variable")
drivers["bear spread"] = drivers["P(WTI bear next | var Bear)"] - drivers["P(WTI bear next | var Bull)"]
show_table(drivers.sort_values("bear spread", ascending=False), 7, "driver_regimes_vs_wti_next", "Training block: probability of a bear or bull WTI month ahead given the current regime of each driver (sorted by how much the driver's regime shifts the bear probability)", decimals=3)

## Summary and Exports

Every table and figure produced above has been saved under `GWP2_Outputs/`:

- **Step 1**: the toy example (posterior figure, Baum-Welch recovery and convergence, decoding summary);
- **Step 2**: the WTI regime figure under Alvi's method and the GWP2 method, the HMM parameters of each, the fit stability, the episode summary and the bull, bear and stagnant episode tables;
- **Step 4**: the sample split, the variables in scope, the panel-wide HMM stability and warning flags under both regime methods, the regime plots, the Cramer's V map, the discretised regime tables, the full hill-climb runs, the forecast node's CPDs, the validation and test scores (with and without the current WTI regime as evidence), the confusion matrices, the turning-point accuracy, the bootstrap arc stability;
- **Step 6**: the transformation chain, the symbol mix, the Baum-Welch trace and the Viterbi against posterior figure;
- **Step 7**: the latent meaning tables;
- **Step 9**: the minimal working example of hill climbing.

The final cell bundles everything into `GWP2_All_Outputs.zip`.

In [ ]:
zip_name = "GWP2_All_Outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT):
        for f in sorted(files):
            zf.write(os.path.join(root, f))
n_files = sum(len(files) for _, _, files in os.walk(OUT))
print(f"{n_files} files written to {OUT}/ and bundled into {zip_name} ({os.path.getsize(zip_name) / 1e6:.1f} MB).")
print(f"Figures: {FIG_NO} | Tables: {TAB_NO} | Seed: {SEED}")
try:
    from google.colab import files as colab_files
    colab_files.download(zip_name)
except Exception:
    print("Not running in Colab: the zip file is in the working directory.")